In [1]:
import sys
sys.path.append("../pipelines")

from aq import *
import pandas as pd

print("Connexion prête !")

Connexion prête !


In [4]:
# %% US-01 · Inventaire des sources cif et crm

# 1.1 Tables, colonnes et clé primaire
cols = lire("""SELECT table_schema AS base,
                      table_name AS `table`,
                      COUNT(*) AS colonnes
               FROM information_schema.columns
               WHERE table_schema IN ('cif','crm')
               GROUP BY table_schema, table_name""")

cles = lire("""SELECT table_schema AS base,
                      table_name AS `table`,
                      GROUP_CONCAT(column_name ORDER BY ordinal_position) AS cle_primaire
               FROM information_schema.key_column_usage
               WHERE table_schema IN ('cif','crm')
               AND constraint_name = 'PRIMARY'
               GROUP BY table_schema, table_name""")

inventaire = cols.merge(cles, how="left", on=["base", "table"])

# 1.2 Comptage exact
inventaire["lignes"] = [
    int(lire(f"SELECT COUNT(*) AS n FROM {b}.{t}").n[0])
    for b, t in zip(inventaire.base, inventaire.table)
]

print(inventaire.sort_values(["base", "table"]).to_string(index=False))

# 1.3 Tables sans clé primaire
print("\nTables sans clé primaire :",
      inventaire[inventaire.cle_primaire.isna()][["base", "table"]].values.tolist())

base        table  colonnes cle_primaire  lignes
 cif      clients        10      column0   40000
 cif       compte         4      column0   45000
 cif         pays         2    code_pays     250
 crm       client        13    client_id  100000
 crm      contrat         9   contrat_id  100000
 crm     sinistre        10  sinistre_id  100000
 crm type_contrat         3    code_type      12

Tables sans clé primaire : []


In [3]:
tables = [
    "crm.client",
    "crm.contrat",
    "crm.sinistre",
    "crm.type_contrat",
    "cif.clients",
    "cif.compte",
    "cif.pays"
]

for table in tables:
    df = lire(f"SELECT COUNT(*) AS nb_lignes FROM {table}")
    print(f"{table} : {df.iloc[0]['nb_lignes']} lignes")

crm.client : 100000 lignes
crm.contrat : 100000 lignes
crm.sinistre : 100000 lignes
crm.type_contrat : 12 lignes
cif.clients : 40000 lignes
cif.compte : 45000 lignes
cif.pays : 250 lignes
